In [77]:
import pandas as pd
import os
from ytmusicapi import YTMusic
import random
import time

# Path to header .json file for ytmusic api
ytmusic_header_path = '..'
# Path to .tsv reddit search logs
reddit_log_path = '..\\..\\reddit-scraper\\logs\\'

def parse_ytmusic_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(
        lambda x: x[0]['id'])  # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_ytmusic_playlist(yt, playlist_meta):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    tracks = parse_ytmusic_tracks(track_list)
    return tracks, playlist_meta
    
# Init yt api
ytm = YTMusic(os.path.join(ytmusic_header_path, 'headers_auth.json'))

# Load db
db_tsv_path = os.path.join(reddit_log_path, 'ytmusic', 'reddit_ytmusic_subreddit_db.tsv')
db = pd.read_csv(db_tsv_path, sep='\t', index_col=0)
print('db manual labels:',db.manual_label.unique())

db_pass = db.loc[db.manual_label == 'passed-match']
print('passing db results types:',db_pass.ytmusic_resultType.unique())
print(len(db.loc[db.ytmusic_videoId == '#NAME?']))
print(len(db.loc[db.youtube_videoId == '#NAME?']))


db manual labels: ['passed-match' 'failed-match' 'no-match']
passing db results types: ['song' 'album']
0
0


## Subreddit playlists for passing tracks

In [79]:
SLEEP_TIME=5
LIMIT = 1000
SHUFFLE_PLAYLIST = True
completed = []
db_pass_tracks = db_pass.loc[db_pass.ytmusic_resultType == 'song']
for sub, df in db_pass_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing tracks')
    vids = df.ytmusic_videoId.unique().tolist()
    if SHUFFLE_PLAYLIST:
        random.shuffle(vids)
    title=f'x_r.{sub}_tracks_db'
    desc = f'Matched {len(vids)} tracks from r.{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=list(vids))
    print(f'Saved {len(vids)} r.{sub} tracks playlist with id: {pl_id}, waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    if len(tracks) < 10:
        print(f'Not enough tracks to split into like dislike: count = {len(tracks)}')
        completed.append(sub)
        continue

    # Create Like subset
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    vids = liked_tracks.videoId.unique().tolist()
    desc = f'Liked subset of {len(vids)} entries from: {desc}'
    liked_pl_id = ytm.create_playlist(title=f'{title}_like', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} LIKE r.{sub} tracks playlist with id: {liked_pl_id}, waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)
    
    # Create Unrated Radio subset
    unrated_tracks = tracks.loc[tracks['likeStatus'] != 'LIKE']
    vids = unrated_tracks.videoId.unique().tolist()
    desc = f'Unrated radio subset of {len(vids)} entries from: {desc}'
    radio_pl_id = ytm.create_playlist(title=f'{title}_radio', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} INDIFFERENT r.{sub} radio tracks playlist with id: {liked_pl_id}, waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)

    # Delete original playlist now thet like/unrated is split
    ytm.delete_playlist(pl_id)
    print(f'Deleted r/{sub} tracks playlist with id: {pl_id}')
    completed.append(sub)
    



Generating r.90shiphop ytmusic playlist for 641 passing tracks
Saved 404 r.90shiphop tracks playlist with id: PLWptjpDqazOylW78qnH03dpID2XAj05mD, waiting 5 seconds...
Filtered 203 LIKE r.90shiphop tracks playlist with id: PLWptjpDqazOyEC5u0Z10hcrPfM16171Ab, waiting 5 seconds...
Filtered 201 INDIFFERENT r.90shiphop radio tracks playlist with id: PLWptjpDqazOyEC5u0Z10hcrPfM16171Ab, waiting 5 seconds...
Deleted r/90shiphop tracks playlist with id: PLWptjpDqazOylW78qnH03dpID2XAj05mD

Generating r.blues ytmusic playlist for 323 passing tracks
Saved 276 r.blues tracks playlist with id: PLWptjpDqazOwEgJ_-z__aQNZzdegwvLgs, waiting 5 seconds...
Filtered 75 LIKE r.blues tracks playlist with id: PLWptjpDqazOz4OKNHI3JB5b5C-dbCRKav, waiting 5 seconds...
Filtered 201 INDIFFERENT r.blues radio tracks playlist with id: PLWptjpDqazOz4OKNHI3JB5b5C-dbCRKav, waiting 5 seconds...
Deleted r/blues tracks playlist with id: PLWptjpDqazOwEgJ_-z__aQNZzdegwvLgs

Generating r.chillmusic ytmusic playlist for 665 p

## Subreddit playlists for passing albums


In [22]:
SLEEP_TIME=5
LIMIT = 1000
completed= []
db_pass_albums = db_pass.loc[db_pass.ytmusic_resultType == 'album']

for sub, df in db_pass_albums.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing albums')
    vids = list()
    for row in df.itertuples():
        print(f' Adding album: {row.ytmusic_album}')
        try:
            for track in  ytm.get_album(row.ytmusic_albumId).get('tracks', []):
                if track['videoId'] not in vids:
                    vids.append(track['videoId'])
        except Exception as e:
            print(e)
            print('skipping', row.ytmusic_album)

    title=f'x_r.{sub}_albums_db'
    desc = f'Matched {len(vids)} albums from r.{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
    print(f'Saved {len(vids)} r.{sub} albums playlist with id: {pl_id}, waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)


    if len(tracks) < 10:
        print(f'Not enough tracks to split into like dislike: count = {len(tracks)}')
        completed.append(sub)
        continue

    # Create Like subset to merge with tracks playlist
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    vids = liked_tracks.videoId.unique().tolist()
    desc = f'Liked subset of {len(vids)} entries from: {desc}'
    liked_pl_id = ytm.create_playlist(title=f'{title}_tracks_like', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} LIKE r.{sub} tracks playlist with id: {liked_pl_id}, waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)
    completed.append(sub)




Generating r.psychedelicrock ytmusic playlist for 52 passing albums
 Adding album: Allah-Las
 Adding album: Dandelion Gum
 Adding album: Jassbusters
 Adding album: The Man Who Sold the World
 Adding album: Sunshine Superman
 Adding album: Gris Gris
 Adding album: Driving Under the Influence of Jams
 Adding album: ZAM
 Adding album: Maggot Brain
Server returned HTTP 404: Not Found.
Requested entity was not found.
skipping Maggot Brain
 Adding album: Space Ritual
 Adding album: Hawkwind
 Adding album: Space Ritual
 Adding album: Volunteers
 Adding album: Masana Temples
 Adding album: House in the Tall Grass
 Adding album: Flying Microtonal Banana
 Adding album: I'm In Your Mind Fuzz
 Adding album: Gumboot Soup
 Adding album: Polygondwanaland
 Adding album: Psycho Tropical Berlin
 Adding album: Melody's Echo Chamber
 Adding album: Melody's Echo Chamber
 Adding album: Bon Voyage
 Adding album: Bon Voyage
 Adding album: Mink Mussel Manticore
 Adding album: Morgan Delt
 Adding album: Cyberf

In [ ]:
## one time fix for ?name video ids FUCK EXecl
# print(len(db.loc[db.ytmusic_videoId == '#NAME?']))


# fixed_ids = []
# for row in db.loc[db.ytmusic_videoId == '#NAME?'].itertuples():
#     query = f'{row.ytmusic_title} {row.ytmusic_artist} {row.ytmusic_album}'
#     res = ytm.search(query=query, filter='songs', limit=5)
#     if not len(res):
#         res = ytm.search(query=f'{row.ytmusic_title} {row.ytmusic_artist}', filter='songs', limit=3)
#         if not len(res):
#             print('\n\nNo result!', query, row.manual_label)
#             fixed_ids.append(None)
#             continue
#     trys = []
#     vid = None
#     for r in res:
#         albumName = r.get('album', '').get('name', '')
#         albumId = r.get('album', '').get('id', '')
#         trys.append(albumName)
#         if row.ytmusic_albumId == albumId:
#             vid = r['videoId']
#             break
#         elif  row.ytmusic_album.lower() in albumName.lower():
#             vid = r['videoId']
#             break
#     if not vid:
#         print('\n\n',row.ytmusic_key, row.ytmusic_album, row.manual_label, '!=', trys)
#     fixed_ids.append(vid) 

# db.loc[db['ytmusic_videoId'] == '#NAME?', 'ytmusic_videoId'] = fixed_ids
# print(len(db.loc[db.ytmusic_videoId == '#NAME?']))
# db.loc[db['youtube_videoId'] == '#NAME?', 'youtube_videoId'] = None

# db_file = os.path.join(
#     reddit_log_path, 'ytmusic', f'reddit_ytmusic_subreddit_db.tsv')
# db.to_csv(db_file, sep='\t', header=True)